In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Використовуємо відфільтрований датафрейм (без tvod == 1)
user_df = pd.read_csv(r'C:\Users\pinko\Documents\dell files\desktop\analytical project\data\user_df_sample.csv')
cluster_df = user_df.drop(columns=['IOS']).copy()

# Обираємо числові ознаки для кластеризації
# (Виключаємо ID та константні колонки, де лише одне унікальне значення)
features = cluster_df.select_dtypes(include=['number']).copy()
features = features.loc[:, features.nunique() > 1]

# Заповнюємо можливі пропуски нулями
features = features.fillna(0)


# Стандартизація даних

features_log = np.log1p(features.clip(lower=0))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_log)

OPTIMAL_K = 4
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
cluster_df['cluster'] = kmeans.fit_predict(X_scaled)

print(f"\nРозподіл користувачів по {OPTIMAL_K} кластерах:")
print(cluster_df['cluster'].value_counts().sort_index())

# Виводимо середні показники по кожному кластеру для інтерпретації
cluster_profile = cluster_df.groupby('cluster')[features.columns].mean().T
print("\nСередні значення ознак у кожному кластері:")
print(cluster_profile.round(2))

#  Візуалізація кластерів
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
cluster_df['pca_1'] = coords[:, 0]
cluster_df['pca_2'] = coords[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=cluster_df,
    x='pca_1',
    y='pca_2',
    hue='cluster',
    palette='tab10',
    alpha=0.6,
    s=30
)
plt.title(f'Візуалізація {OPTIMAL_K} кластерів користувачів (PCA)', fontsize=14)

plt.legend(title='Кластер')
plt.grid(True, alpha=0.3)
plt.savefig(r'C:\Users\pinko\Documents\dell files\desktop\analytical project\images\clusters.png', dpi=150, bbox_inches='tight')
plt.show()
